# 🏙️ SmartCity: MobileNetV3 Civic Image Classifier Training
### End-to-End Transfer Learning & Evaluation Pipeline for Google Colab

This notebook trains and fine-tunes **MobileNetV3 Small** for civic complaints (`Waste Management`, `Roads`, `Drainage`, `Water Supply`, `Other`).

**Features:**
- 🚀 Automatic GPU acceleration (CUDA / T4)
- 🎯 Two-phase transfer learning: Head-only warmup, followed by deep fine-tuning
- 📊 Exact metric computation: Accuracy, Macro Precision/Recall/F1, Confusion Matrix
- 📦 Exports `best_model.pth` and `model_config.json` ready for deployment in `ai/models/`

In [ ]:
# 1. Environment & GPU Check
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Active Device: {device}")

In [ ]:
# 2. Install dependencies
!pip install -q torch torchvision scikit-learn matplotlib pillow

In [ ]:
# 3. Setup Dataset Directory Structure
import os

DATA_DIR = './data'
CATEGORIES = ['Waste Management', 'Roads', 'Drainage', 'Water Supply', 'Other']

for cat in CATEGORIES:
    os.makedirs(os.path.join(DATA_DIR, cat), exist_ok=True)

print("Upload your categorized images into the corresponding folders in ./data/")
print("Or unzip your dataset: !unzip civic_dataset.zip -d ./data")

In [ ]:
# 4. Define MobileNetV3 Small Model Architecture
import torch.nn as nn
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights

def build_model(num_classes=5, pretrained=True):
    weights = MobileNet_V3_Small_Weights.DEFAULT if pretrained else None
    model = mobilenet_v3_small(weights=weights)
    in_features = model.classifier[3].in_features
    model.classifier[2] = nn.Dropout(p=0.2, inplace=True)
    model.classifier[3] = nn.Linear(in_features=in_features, out_features=num_classes)
    return model

def freeze_backbone(model):
    for param in model.features.parameters():
        param.requires_grad = False
    for param in model.classifier.parameters():
        param.requires_grad = True

def unfreeze_backbone(model, last_n_blocks=3):
    children = list(model.features.children())
    for child in children[-last_n_blocks:]:
        for param in child.parameters():
            param.requires_grad = True

print("Model builder defined.")

In [ ]:
# 5. Data Transforms & Stratified Split
from torchvision import transforms
from torchvision.transforms import InterpolationMode
from sklearn.model_selection import train_test_split
from PIL import Image
from torch.utils.data import Dataset, DataLoader

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((256, 256), interpolation=InterpolationMode.BILINEAR),
    transforms.RandomResizedCrop(224, scale=(0.75, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

eval_transform = transforms.Compose([
    transforms.Resize((256, 256), interpolation=InterpolationMode.BILINEAR),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

class CivicDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        fpath, label = self.samples[idx]
        img = Image.open(fpath).convert('RGB')
        return self.transform(img) if self.transform else transforms.ToTensor()(img), label

print("Dataset utilities configured.")

In [ ]:
# 6. Scan Data, Create Train/Val/Test Partitions, and DataLoaders
samples = []
class_to_idx = {cat: idx for idx, cat in enumerate(CATEGORIES)}

for cat in CATEGORIES:
    cdir = os.path.join(DATA_DIR, cat)
    if os.path.exists(cdir):
        for fname in os.listdir(cdir):
            if os.path.splitext(fname)[1].lower() in ['.jpg', '.jpeg', '.png', '.webp']:
                samples.append((os.path.join(cdir, fname), class_to_idx[cat]))

print(f"Total images collected: {len(samples)}")

paths = [s[0] for s in samples]
labels = [s[1] for s in samples]

train_p, test_val_p, train_l, test_val_l = train_test_split(
    paths, labels, test_size=0.30, stratify=labels, random_state=42
)
val_p, test_p, val_l, test_l = train_test_split(
    test_val_p, test_val_l, test_size=0.50, stratify=test_val_l, random_state=42
)

train_set = CivicDataset(list(zip(train_p, train_l)), transform=train_transform)
val_set = CivicDataset(list(zip(val_p, val_l)), transform=eval_transform)
test_set = CivicDataset(list(zip(test_p, test_l)), transform=eval_transform)

train_loader = DataLoader(train_set, batch_size=16, shuffle=True)
val_loader = DataLoader(val_set, batch_size=16, shuffle=False)
test_loader = DataLoader(test_set, batch_size=16, shuffle=False)
print(f"Splits -> Train: {len(train_set)}, Val: {len(val_set)}, Test: {len(test_set)}")

In [ ]:
# 7. Two-Phase Training Loop with Validation Checkpointing
import time
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

model = build_model(num_classes=len(CATEGORIES), pretrained=True).to(device)
criterion = nn.CrossEntropyLoss()

best_val_loss = float('inf')
best_weights = None

# PHASE 1: Head Only
freeze_backbone(model)
optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=5)

print("Starting Phase 1 (Frozen Backbone)...")
for ep in range(1, 6):
    model.train()
    t_loss, correct, total = 0.0, 0, 0
    for imgs, tgts in train_loader:
        imgs, tgts = imgs.to(device), tgts.to(device)
        optimizer.zero_grad()
        outs = model(imgs)
        loss = criterion(outs, tgts)
        loss.backward()
        optimizer.step()
        t_loss += loss.item() * imgs.size(0)
        _, preds = torch.max(outs, 1)
        correct += torch.sum(preds == tgts).item()
        total += tgts.size(0)
    scheduler.step()
    
    # Val
    model.eval()
    v_loss, v_corr, v_tot = 0.0, 0, 0
    with torch.no_grad():
        for imgs, tgts in val_loader:
            imgs, tgts = imgs.to(device), tgts.to(device)
            outs = model(imgs)
            v_loss += criterion(outs, tgts).item() * imgs.size(0)
            _, preds = torch.max(outs, 1)
            v_corr += torch.sum(preds == tgts).item()
            v_tot += tgts.size(0)
    
    v_loss /= max(v_tot, 1)
    print(f"[Phase 1 Ep {ep}/5] Train Loss: {t_loss/total:.4f} Acc: {correct/total*100:.1f}% | Val Loss: {v_loss:.4f} Acc: {v_corr/v_tot*100:.1f}%")
    if v_loss < best_val_loss:
        best_val_loss = v_loss
        best_weights = {k: v.cpu().clone() for k, v in model.state_dict().items()}

# PHASE 2: Deep Fine-Tuning
unfreeze_backbone(model, last_n_blocks=3)
optimizer = AdamW([
    {'params': model.features.parameters(), 'lr': 2e-5},
    {'params': model.classifier.parameters(), 'lr': 1e-4}
], weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=5)

print("\nStarting Phase 2 (Fine-Tuning Deeper Layers)...")
for ep in range(1, 6):
    model.train()
    t_loss, correct, total = 0.0, 0, 0
    for imgs, tgts in train_loader:
        imgs, tgts = imgs.to(device), tgts.to(device)
        optimizer.zero_grad()
        outs = model(imgs)
        loss = criterion(outs, tgts)
        loss.backward()
        optimizer.step()
        t_loss += loss.item() * imgs.size(0)
        _, preds = torch.max(outs, 1)
        correct += torch.sum(preds == tgts).item()
        total += tgts.size(0)
    scheduler.step()
    
    # Val
    model.eval()
    v_loss, v_corr, v_tot = 0.0, 0, 0
    with torch.no_grad():
        for imgs, tgts in val_loader:
            imgs, tgts = imgs.to(device), tgts.to(device)
            outs = model(imgs)
            v_loss += criterion(outs, tgts).item() * imgs.size(0)
            _, preds = torch.max(outs, 1)
            v_corr += torch.sum(preds == tgts).item()
            v_tot += tgts.size(0)
    
    v_loss /= max(v_tot, 1)
    print(f"[Phase 2 Ep {ep}/5] Train Loss: {t_loss/total:.4f} Acc: {correct/total*100:.1f}% | Val Loss: {v_loss:.4f} Acc: {v_corr/v_tot*100:.1f}%")
    if v_loss < best_val_loss:
        best_val_loss = v_loss
        best_weights = {k: v.cpu().clone() for k, v in model.state_dict().items()}

In [ ]:
# 8. Test Set Evaluation & Confusion Matrix
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

model.load_state_dict(best_weights)
model.eval()

y_true, y_pred = [], []
with torch.no_grad():
    for imgs, tgts in test_loader:
        outs = model(imgs.to(device))
        _, preds = torch.max(outs, 1)
        y_true.extend(tgts.numpy().tolist())
        y_pred.extend(preds.cpu().numpy().tolist())

acc = accuracy_score(y_true, y_pred)
print(f"\nFINAL TEST ACCURACY: {acc*100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=CATEGORIES, zero_division=0))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(7, 5))
plt.imshow(cm, cmap='Blues')
plt.colorbar()
plt.xticks(range(len(CATEGORIES)), CATEGORIES, rotation=45, ha='right')
plt.yticks(range(len(CATEGORIES)), CATEGORIES)
for i in range(len(CATEGORIES)):
    for j in range(len(CATEGORIES)):
        plt.text(j, i, str(cm[i, j]), ha='center', va='center')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Test Confusion Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# 9. Export Artifacts for SmartCity Microservice
import json
from google.colab import files

os.makedirs('./models', exist_ok=True)
torch.save(best_weights, './models/best_model.pth')

config = {
    "model_name": "MobileNetV3-Small-Civic",
    "architecture": "mobilenet_v3_small",
    "num_classes": len(CATEGORIES),
    "classes": CATEGORIES,
    "class_to_idx": class_to_idx,
    "image_size": 224,
    "input_mean": IMAGENET_MEAN,
    "input_std": IMAGENET_STD
}
with open('./models/model_config.json', 'w') as f:
    json.dump(config, f, indent=2)

print("Artifacts generated in ./models/")
print("Download best_model.pth and model_config.json, then place them in smart-city/ai/models/")
files.download('./models/best_model.pth')
files.download('./models/model_config.json')